# Zero-shot ROCOv2 captioning --- Qwen3-VL **V1** (radiology-constrained prompt)

Same model and pipeline as `qwen3vl_roco_zeroshot_v0.ipynb`, but with a prompt **adapted
to this instruction-tuned model** instead of the bare `"a photo of"`:

* a **system** turn (expert radiologist; describe only what is visible; terse clinical
  register; no preamble), plus
* a **user** turn (image + *"Provide the caption for this image."*).

The model answers the instruction normally and we keep its whole reply. Everything else ---
greedy decoding, batch 1, resolution cap, the five-metric suite --- is identical to V0, so the
**only V0 → V1 difference is the prompt**.

> **V1 shows Qwen at its best; it is NOT the strict same-prompt comparison to
> BLIP-2/BioMedVQA (that is V0).** Report both. And run V0 and V1 at the **same**
> `MAX_VIS_TOKENS`, else the V0→V1 gap conflates prompt with resolution.

The full 9,927-image run is best done headless via `qwen3vl_roco_zeroshot_v1.py` in `tmux`
(resumable); this notebook defaults to a 500-image subset.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if device == "cuda" else torch.float32
print("device =", device, "| dtype =", DTYPE)

In [ ]:
# ------------------------------- Config ------------------------------- #
MODEL_ID     = "/home/matei/qwen3-vl-4b-instruct"   # local copy of 4B bf16
LOAD_IN_4BIT = False
MIN_PIXELS   = 256 * 32 * 32
# 1024 = quality-optimal for ROCO; lower to 768/512 for speed. Keep EQUAL to the V0 run.
MAX_PIXELS   = 1024 * 32 * 32

In [ ]:
# ------------------------ Load Qwen3-VL (processor + model) ------------------------ #
from transformers import AutoProcessor, AutoModelForImageTextToText
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "left"

_kwargs = {}
if device == "cuda":
    _kwargs["device_map"] = {"": 0}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    _kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
else:
    _kwargs["dtype"] = DTYPE
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **_kwargs)

# Optional: load a fine-tuned checkpoint (LoRA adapter + trained mergers).
# Leave LORA_PATH = None for the zero-shot run -> identical code path either way,
# so the zero-shot -> fine-tuned delta is clean.
LORA_PATH = None      # e.g. "/home/matei/qwen3vl_ft_ckpt/best"
if LORA_PATH:
    merger_file = os.path.join(LORA_PATH, "merger.pt")
    if os.path.isfile(merger_file):                 # mergers first: plain weights, not LoRA
        msd = torch.load(merger_file, map_location="cpu")
        params = dict(model.named_parameters())
        def _norm(k):                               # strip peft "base_model.model." prefixes
            i = k.find("model.visual")
            return k[i:] if i >= 0 else k
        n_ok = 0
        for k, v in msd.items():
            tgt = params.get(_norm(k))
            if tgt is not None and tgt.shape == v.shape:
                tgt.data.copy_(v.to(tgt.dtype)); n_ok += 1
        print(f"[ft] loaded {n_ok}/{len(msd)} merger tensors")
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, LORA_PATH)
    model = model.merge_and_unload()                # fold LoRA in -> no inference overhead
    print(f"[ft] merged LoRA adapter from {LORA_PATH}")

model.eval()
if device == "cuda":
    print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ----------------- ROCOv2 test split (same loader as caption_roco.py) ----------------- #
import pandas as pd
ROCO_DIR = "/home/matei/rocov2"
IMG_DIR  = os.path.join(ROCO_DIR, "test")
caps = pd.read_csv(os.path.join(ROCO_DIR, "test_captions.csv")).dropna(subset=["Caption"]).reset_index(drop=True)
records = [{"id": r.ID, "path": os.path.join(IMG_DIR, f"{r.ID}.jpg"), "caption": str(r.Caption)}
           for r in caps.itertuples() if os.path.isfile(os.path.join(IMG_DIR, f"{r.ID}.jpg"))]
print(f"ROCOv2 test: {len(records)} image-caption pairs")
print("example:", records[0]["id"], "->", records[0]["caption"][:80])

In [ ]:
# ---------------- V1 prompt: radiology-constrained system turn + instruction ---------------- #
from tqdm.auto import tqdm
SYSTEM_PROMPT = (
    "You are an expert radiologist writing captions for a radiology teaching archive. "
    "You are given one medical image. Write a single concise caption that describes only "
    "what is directly visible.\n"
    "Rules:\n"
    "1. Describe only what is observable in this image: the imaging modality, the anatomical "
    "region, and any clearly visible findings. Do not infer diagnoses, patient history, or "
    "findings that are not directly visible.\n"
    "2. If a finding cannot be determined from the image, do not state it.\n"
    "3. Be terse and clinical: one sentence, radiology-report style.\n"
    "4. Output only the caption -- no preamble, no \"This image shows\", no disclaimers."
)
USER_PROMPT   = "Provide the caption for this image."
CAPTION_BATCH = 1
GEN_KWARGS = dict(max_new_tokens=40, min_new_tokens=8, do_sample=False, num_beams=1,
                  no_repeat_ngram_size=3,
                  pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id)

# Build the prompt text ONCE (system + user identical for every image). Unlike V0 we do NOT
# prime the assistant reply -- the model answers the instruction and we keep its whole reply.
_DUMMY = Image.new("RGB", (32, 32))
_msgs  = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",   "content": [{"type": "image", "image": _DUMMY},
                                   {"type": "text",  "text": USER_PROMPT}]},
]
PROMPT_TEXT = processor.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def caption_records(recs, batch_size=CAPTION_BATCH):
    preds = []
    for i in tqdm(range(0, len(recs), batch_size), desc="captioning", leave=False):
        chunk = recs[i:i + batch_size]
        imgs  = [Image.open(r["path"]).convert("RGB") for r in chunk]   # radiographs -> RGB
        inputs = processor(text=[PROMPT_TEXT] * len(imgs), images=imgs,
                           padding=True, return_tensors="pt").to(model.device)
        gen = model.generate(**inputs, **GEN_KWARGS)
        gen = gen[:, inputs["input_ids"].shape[1]:]        # left pad -> strip shared prompt
        preds.extend(s.strip() for s in processor.batch_decode(gen, skip_special_tokens=True))
    return preds

In [ ]:
# ------------------------- Eyeball: does the constrained prompt help? ------------------------- #
sample = records[:8]
sp = caption_records(sample, batch_size=1)
for r, p in zip(sample, sp):
    print(f"REF : {r['caption'][:95]}")
    print(f"PRED: {p[:95]}\n")

In [ ]:
# ---------------- Metrics: BLEU-1..4, METEOR, ROUGE-L, CIDEr, BERTScore (IDENTICAL to V0) ---------------- #
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score

import transformers.modeling_utils as _mu
_mu.check_torch_load_is_safe = lambda *a, **k: None
from bert_score import BERTScorer
BERT_MODEL = "microsoft/deberta-xlarge-mnli"
_bert_scorer = BERTScorer(model_type=BERT_MODEL, batch_size=8)
_bert_scorer._tokenizer.model_max_length = 512

def _norm(s):
    return " ".join(str(s).lower().split())

def compute_caption_metrics(preds, refs):
    preds = [p if str(p).strip() else "." for p in preds]
    refs  = [r if str(r).strip() else "." for r in refs]
    gts = {i: [_norm(refs[i])]  for i in range(len(refs))}
    res = {i: [_norm(preds[i])] for i in range(len(preds))}
    bleu, _  = Bleu(4).compute_score(gts, res)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    meteor = sum(meteor_score([_norm(refs[i]).split()], _norm(preds[i]).split())
                 for i in range(len(preds))) / len(preds)
    _, _, F = _bert_scorer.score(preds, refs, batch_size=8, verbose=False)
    m = {"BLEU-1": bleu[0], "BLEU-2": bleu[1], "BLEU-3": bleu[2], "BLEU-4": bleu[3],
         "METEOR": meteor, "ROUGE-L": rouge, "CIDEr": cider,
         "BERTScore-F1": F.mean().item(), "BERTScore-model": BERT_MODEL}
    print("=" * 48)
    for k, v in m.items():
        print(f"  {k:16s}: {v:.4f}" if isinstance(v, float) else f"  {k:16s}: {v}")
    print("=" * 48)
    return m

In [ ]:
# ------------------------ Zero-shot V1 evaluation on ROCOv2 test ------------------------ #
SUBSET_N = 500      # None for the full 9,927 (better run headless via qwen3vl_roco_zeroshot_v1.py)

eval_recs = records if SUBSET_N is None else records[:SUBSET_N]
print(f"captioning {len(eval_recs)} images (V1 radiology-constrained prompt) ...")
preds = caption_records(eval_recs)
refs  = [r["caption"] for r in eval_recs]

print("\nQwen3-VL V1 ZERO-SHOT CAPTIONING -- ROCOv2 test")
metrics = compute_caption_metrics(preds, refs)

print("\nexamples:")
for r, p in list(zip(eval_recs, preds))[:5]:
    print(f"  REF : {r['caption'][:100]}")
    print(f"  PRED: {p[:100]}\n")

### Reading V1

* Compare these captions to the V0 (`"a photo of"`) run **at the same resolution**. The V0→V1
  gap is your "how much does an adapted prompt help this instruct model" result.
* Watch whether the constraint prompt actually suppresses the verbose prose / hedging /
  invented findings that a general model tends to produce (your §6 domain-gap analysis).
* Headline table should show **both** V0 and V1 for Qwen, alongside the (greedy-rescored)
  BLIP-2 / BioMedVQA numbers on the ImageCLEF BERTScore.